# SceneFit benchmark on Colab

This notebook runs the benchmark in three resumable stages: prepare, judge, and retrieve. Checkpoints are copied to Google Drive after every completed scene. The default is CLIP plus Aesthetic on a T4; change `METHODS` only before beginning a fresh checkpoint directory.

Uvicorn is deliberately started from the Colab terminal between the judge and retrieve cells, so its model-loading logs remain visible.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/SceneFit-Backend2')
REPOSITORY_URL = 'https://github.com/dihy16/SceneFit-Backend2.git'
REPOSITORY_BRANCH = 'main'
DATA_ARCHIVE = Path('/content/drive/MyDrive/VRetrieval/data.zip')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/VRetrieval/benchmark/checkpoints-light-gemma4')
NUM_SCENES = 10
NUM_OUTFITS = 100
SEED = 42
JUDGE_MODEL = 'gemma-4-31b-it'
BATCH_SIZE = 10
METHODS = ['clip', 'aesthetic']  # Use a fresh checkpoint directory if changed.

os.environ.update({
    'PROJECT_ROOT': str(PROJECT_ROOT),
    'REPOSITORY_URL': REPOSITORY_URL,
    'REPOSITORY_BRANCH': REPOSITORY_BRANCH,
    'DATA_ARCHIVE': str(DATA_ARCHIVE),
    'CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'NUM_SCENES': str(NUM_SCENES),
    'NUM_OUTFITS': str(NUM_OUTFITS),
    'SEED': str(SEED),
    'JUDGE_MODEL': JUDGE_MODEL,
    'BATCH_SIZE': str(BATCH_SIZE),
    'METHODS': ' '.join(METHODS),
})
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Methods: {METHODS}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
%%bash
set -euo pipefail

if [ -d "$PROJECT_ROOT/.git" ]; then
  git -C "$PROJECT_ROOT" fetch origin "$REPOSITORY_BRANCH"
  git -C "$PROJECT_ROOT" pull --ff-only origin "$REPOSITORY_BRANCH"
else
  git clone --branch "$REPOSITORY_BRANCH" --recurse-submodules "$REPOSITORY_URL" "$PROJECT_ROOT"
fi

python -m pip install -r "$PROJECT_ROOT/requirements.txt"
python -m pip install 'qwen-vl-utils>=0.0.14' 'rembg[gpu]' faiss-cpu

if [ ! -d "$PROJECT_ROOT/data/2d" ]; then
  test -f "$DATA_ARCHIVE"
  unzip -q "$DATA_ARCHIVE" -d "$PROJECT_ROOT"
fi
mkdir -p "$CHECKPOINT_DIR"
cd "$PROJECT_ROOT"
git rev-parse --short HEAD

In [ ]:
from google.colab import userdata

def read_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f'Add {name} in Colab Secrets, then rerun this cell.')
    return value

gemini_key = read_secret('GEMINI_API_KEY', required=True)
hf_token = read_secret('HF_TOKEN')
imagerouter_key = read_secret('IMAGEROUTER_API_KEY') or 'unused-for-light-benchmark'
env_lines = [
    f'GEMINI_API_KEY={gemini_key}',
    f'IMAGEROUTER_API_KEY={imagerouter_key}',
]
if hf_token:
    env_lines.append(f'HF_TOKEN={hf_token}')
env_path = PROJECT_ROOT / '.env'
env_path.write_text('\n'.join(env_lines) + '\n', encoding='utf-8')
env_path.chmod(0o600)
os.environ['GEMINI_API_KEY'] = gemini_key
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('Colab Secrets loaded without printing their values.')

## Optional: upload missing background images

Run the next cell only when `data.zip` does not already contain at least `NUM_SCENES` files in `data/bg`. Select the missing image files in the upload dialog.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# target = PROJECT_ROOT / 'data' / 'bg'
# target.mkdir(parents=True, exist_ok=True)
# for filename, content in uploaded.items():
#     (target / Path(filename).name).write_bytes(content)
# print(f'Background images available: {len(list(target.iterdir()))}')

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_ROOT"
test "$(find data/bg -maxdepth 1 -type f | wc -l)" -ge "$NUM_SCENES"
python scripts/benchmark.py smoke

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_ROOT"
python scripts/run_benchmark_runtime.py prepare \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --num-scenes "$NUM_SCENES" --num-outfits "$NUM_OUTFITS" --seed "$SEED" \
  --model "$JUDGE_MODEL" --batch-size "$BATCH_SIZE" --methods $METHODS

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_ROOT"
python scripts/run_benchmark_runtime.py judge \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --num-scenes "$NUM_SCENES" --num-outfits "$NUM_OUTFITS" --seed "$SEED" \
  --model "$JUDGE_MODEL" --batch-size "$BATCH_SIZE" --methods $METHODS

## Start Uvicorn in the Colab terminal

Open **Terminal** in Colab and leave this command running. It is the same shell command used by the notebook's retrieval step:

```bash
cd /content/SceneFit-Backend2
BENCHMARK_MANIFEST=/content/SceneFit-Backend2/results/benchmark/latest/manifest.json \
  python -m uvicorn app.main:app --host 0.0.0.0 --port 8000
```

Wait for `Application startup complete` before running the next cell. The worker configuration uses `http://127.0.0.1:8000`, so Ngrok is not needed.

In [ ]:
import time
import httpx

for attempt in range(60):
    try:
        response = httpx.get('http://127.0.0.1:8000/openapi.json', timeout=5)
        response.raise_for_status()
        print('Uvicorn is ready.')
        break
    except Exception:
        if attempt == 59:
            raise RuntimeError('Uvicorn is not reachable. Check the Colab terminal logs.')
        time.sleep(5)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_ROOT"
python scripts/run_benchmark_runtime.py retrieve \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --num-scenes "$NUM_SCENES" --num-outfits "$NUM_OUTFITS" --seed "$SEED" \
  --model "$JUDGE_MODEL" --batch-size "$BATCH_SIZE" --methods $METHODS

In [ ]:
import json
import pandas as pd

summary_path = PROJECT_ROOT / 'results' / 'benchmark' / 'latest' / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(pd.DataFrame(summary['metrics']).T)
print(f"Final archive: {CHECKPOINT_DIR / 'final.zip'}")